# 反向传播算法

按代码单元格的原编号查阅课程手册。

In [ ]:
# 导入模块
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import ListedColormap

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=2000, noise=0.4, random_state=42)

In [ ]:
X

In [ ]:
len(X)

In [ ]:
y

In [ ]:
# 将数据集可视化
plt.figure(figsize=(8, 8))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#B22222', '#87CEFA']), edgecolors='k')

In [ ]:
# 60%作为训练集，40%作为测试集
from sklearn.model_selection import train_test_split
trainX, testX, trainY, testY = train_test_split(X, y, test_size=0.4, random_state=32)

In [ ]:
trainX.shape, trainY.shape, testX.shape, testY.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
# 初始化一个标准化器的实例
standard = StandardScaler()

# 对训练集进行标准化，它会计算训练集的均值和标准差保存起来
trainX = standard.fit_transform(trainX)

# 使用标准化器在训练集上的均值和标准差，对测试集进行归一化
testX = standard.transform(testX)

In [ ]:
# 定义神经网络
# 我们选用较简单的神经网络，它只有两个参数，W（权重矩阵）和 B（偏置矩阵）
def _init_(q):
    # q：W的维度(也就是有 q 个神经元)
    # 我们将 W 与 B 随机初始化
    np.random.seed(3)
    W = np.random.normal(size = (q,)) * 0.05
    B = np.zeros(1,)
    return W, B

In [ ]:
W, B = _init_(5)
W, B

In [ ]:
# 激活函数 sigmoid
from scipy.special import expit
def sigmoid(X):
    # X: np.ndarray， 待激活值
    # sigmoid(x) = 1/(1+exp(-x))
    return expit(X)    

In [ ]:
# 前向传播
def forward(X, W, B):
    # X: np.ndarray, 输入数据, 维度 = (p,q)
    # W：np.ndarray, 权重矩阵, 维度 = (q,)
    # B: np.ndarray, 偏置项， 维度 = (1,)
    # 计算Z = X*W + B
    # predY = sigmoid(Z)

    Z = np.dot(X,W) + B
    predY = sigmoid(Z)
    return predY

In [ ]:
def loss_func(trueY, predY):
    eps = np.finfo(np.float64).eps
    prob = np.clip(predY, eps, 1.0 - eps)
    return -np.mean(trueY * np.log(prob) + (1 - trueY) * np.log1p(-prob))

In [ ]:
# 反向传播：包括2个部分
def backword(W, B, trueY, predY, X, learning_rate):
    # W: np.ndarray, 权重矩阵, 维度 = (q,)
    # B: np.ndarray, 偏置项， 维度 = (1,)
    # trueY: np.ndarray, 真值， 维度 = (p, )
    # predY: np.ndarray, 预测值， 维度 = (p, )
    # X: np.ndarray, 输入数据, 维度 = (p,q)
    # learning_rate: float， 学习率

    # 1、计算梯度
    # dW: np.ndarray, 损失函数对W的偏导, 维度 = (q,)

    dW = np.dot(X.T, predY - trueY) / len(trueY)
    # dB: float, 损失函数对B的偏导
    dB = np.sum(predY - trueY) / len(trueY)
    

    # 2、参数更新    
    # W = W - learning_rate * dW
    W -= learning_rate * dW
    B -= learning_rate * dB
    

In [ ]:
learning_rate = 0.05

In [ ]:
def train(trainX, trainY, testX, testY, W, B, epochs, flag):
    # trainX: np.ndarray, 训练集, 维度 = (p,q)
    # trainY: np.ndarray, 训练集标签， 维度 = (p, )
    # testX: np.ndarray, 测试集, 维度 = (m,q)
    # testY: np.ndarray, 测试集标签， 维度 = (m, )
    # W：np.ndarray, 权重矩阵, 维度 = (q,)
    # B: np.ndarray, 偏置项， 维度 = (1,)
    # epochs: 迭代次数
    # flag: flag == True 打印损失值\\ flag == False不打印损失值

    train_loss_list = []
    test_loss_list = []
    for i in range(epochs):
        # 训练集
        pred_train_Y = forward(trainX, W, B)
        train_loss = loss_func(trainY, pred_train_Y)
        
        # 验证集
        pred_test_Y = forward(testX, W, B)
        test_loss = loss_func(testY, pred_test_Y)
        
        if flag == True:
            print('the traing loss of %s epoch : %s'%(i+1, train_loss))
            print('the test loss of %s epoch : %s'%(i+1, test_loss))
            print('=========================')
        
        train_loss_list.append(train_loss)
        test_loss_list.append(test_loss)
        
        #反向传播
        backword(W, B, trainY, pred_train_Y, trainX, learning_rate)
    return train_loss_list, test_loss_list

In [ ]:
def predict(X, W, B):
    # X: np.ndarray, 训练集, 维度 = (n, m)
    # W: np.ndarray, 参数, 维度 = (m, 1)
    # B: np.ndarray, 参数b, 维度 = (1, )
    
    y_pred = forward(X,W,B)
    n = len(y_pred)
    prediction = np.zeros((n,1))
    for i in range(n):
        if y_pred[i] > 0.5:
            prediction[i,0] = 1
        else:
            prediction[i,0] = 0
    return prediction

In [ ]:
W, B = _init_(trainX.shape[1])
train_loss_list, test_loss_list = train(trainX, trainY, testX, testY, W, B, 300, False)
pred = predict(testX, W, B)

In [ ]:
from sklearn.metrics import accuracy_score
print(f"Test accuracy: {accuracy_score(testY, pred.ravel()):.4%}")
print(f"Train loss: {train_loss_list[0]:.6f} -> {train_loss_list[-1]:.6f}")

In [ ]:
# 将测试集可视化
plt.figure(figsize = (8, 8))
plt.scatter(testX[:, 0], testX[:, 1], c=np.squeeze(np.array(pred)), cmap=ListedColormap(['#B22222', '#87CEFA']), edgecolors='k')

In [ ]:
def plot_loss(train_loss_list, test_loss_list):
    plt.figure(figsize = (10,8))
    plt.plot(train_loss_list, label='train_loss')
    plt.plot(test_loss_list, label='test_loss')
    
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()

In [ ]:
plot_loss(train_loss_list, test_loss_list)